# Macro Data EDA: Fundamental Vertical

**Author:** Jan  
**Last updated:** 2026-05-16

Exploratory analysis of the 10 starter macro series for the fundamental signal vertical. The goal is to sanity-check coverage, frequencies, and structure of inputs that will feed asset-specific signals on ACWI, AGG, GLD, and BSV. Final signal logic lives in `src/`; this notebook is for inspection and hypothesis generation.

## Setup

In [ ]:
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

project_root = Path.cwd()
if project_root.name == "notebooks":
    project_root = project_root.parent
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from src.config import TRAIN_END
from src.data import load_fred_series

pd.options.display.float_format = "{:.4f}".format
np.random.seed(0)

## Data

Starter set of 10 series matched to the four-ETF mapping:

| Driver | FRED series | Maps to |
|---|---|---|
| 10Y Treasury yield | `DGS10` | AGG, GLD |
| 2Y Treasury yield | `DGS2` | BSV, AGG |
| 10Y real yield (TIPS) | `DFII10` | GLD (primary), AGG |
| 10Y inflation breakeven | `T10YIE` | AGG, GLD |
| Fed funds rate | `FEDFUNDS` | BSV, AGG |
| Core CPI level | `CPILFESL` | inflation level |
| Industrial production | `INDPRO` | ACWI (growth) |
| Nonfarm payrolls | `PAYEMS` | ACWI (growth) |
| Unemployment rate | `UNRATE` | ACWI (growth) |
| Broad USD index | `DTWEXBGS` | GLD |

**Filter to training period.** All EDA in this notebook is restricted to `start <= TRAIN_END` (2024-12-31) so we don't peek at validation (2025) or test (2026+) data. Production code in `src/` can still query any date.

In [ ]:
SERIES = {
    "DGS10": "10Y Treasury yield",
    "DGS2": "2Y Treasury yield",
    "DFII10": "10Y real yield (TIPS)",
    "T10YIE": "10Y inflation breakeven",
    "FEDFUNDS": "Fed funds rate",
    "CPILFESL": "Core CPI level",
    "INDPRO": "Industrial production",
    "PAYEMS": "Nonfarm payrolls",
    "UNRATE": "Unemployment rate",
    "DTWEXBGS": "Broad USD index",
}

In [ ]:
data = {sid: load_fred_series(sid, start="1990-01-01", end=str(TRAIN_END.date())) for sid in SERIES}
df = pd.concat(data, axis=1)
df.tail()

In [ ]:
coverage = pd.DataFrame({
    "description": pd.Series(SERIES),
    "first_obs": df.apply(lambda s: s.first_valid_index()),
    "last_obs": df.apply(lambda s: s.last_valid_index()),
    "n_obs": df.notna().sum(),
})
coverage

## Analysis

In [ ]:
fig, axes = plt.subplots(5, 2, figsize=(14, 14))
for ax, (sid, label) in zip(axes.flat, SERIES.items()):
    df[sid].dropna().plot(ax=ax, linewidth=1)
    ax.set_title(f"{sid}: {label}")
    ax.set_xlabel("")
    ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# Rough correlation of monthly first differences. Mixed units (rates in bp,
# index series in absolute change), so treat as orientation, not truth.
monthly_changes = df.resample("ME").last().diff()
correlation_matrix = monthly_changes.corr()

fig, ax = plt.subplots(figsize=(10, 8))
im = ax.imshow(correlation_matrix, cmap="RdBu_r", vmin=-1, vmax=1, aspect="auto")
ax.set_xticks(range(len(correlation_matrix)))
ax.set_yticks(range(len(correlation_matrix)))
ax.set_xticklabels(correlation_matrix.columns, rotation=45, ha="right")
ax.set_yticklabels(correlation_matrix.columns)
for i in range(len(correlation_matrix)):
    for j in range(len(correlation_matrix)):
        value = correlation_matrix.iloc[i, j]
        ax.text(j, i, f"{value:.2f}", ha="center", va="center",
                fontsize=8, color="white" if abs(value) > 0.5 else "black")
plt.colorbar(im, ax=ax)
plt.title("Correlation of monthly first differences")
plt.tight_layout()
plt.show()

## Results

Populate after running the cells above. Look for: clean coverage with no unexpected gaps, expected leads/lags (e.g., breakevens vs. realized CPI), and which series pairs are strongly correlated vs. independent.

## Notes / next steps

Hypotheses to test in a follow-up notebook:

- **GLD vs. real yields**: regress GLD monthly returns on `DFII10` changes. Expected: negative coefficient.
- **AGG vs. inflation surprises**: build a CPI YoY z-score against its trailing trend; test correlation with AGG forward returns.
- **ACWI vs. growth pulse**: PCA over YoY changes of `INDPRO`, `PAYEMS`, and `UNRATE` to build a single growth factor; test correlation with ACWI forward returns.
- **BSV as the uncertainty sleeve**: when fundamental signals disagree across the other three ETFs, BSV gets weight.

Once hypotheses pass a clean OOS test, productionize the signal logic in `src/fundamental.py` exposing `get_weights(as_of_date) -> pd.Series` per the team contract.